<h1 style="color:green;font-size:22px;">Function 6 - Black-Box Optimisation</h1>
<h1 style="color:#0000CD;font-size:19px;"">Introduction and illustrative analogy</h1>

**Function 6** is a five-dimensional black-box objective over the bounded domain $[0,1]^5$. Its analytical form and the interpretation of its coordinates are unknown. Starting from 20 initial observations, the objective is to identify high-value input configurations through sequential queries under a limited evaluation budget.

An illustrative analogy is the tuning of a machine-learning model with five normalised hyperparameters. The model-performance score can be observed for a selected configuration, but the relationship between the inputs and the score is not available analytically. This analogy is conceptual only: because the challenge does not disclose the meaning of the five coordinates, no domain-specific priors are imposed. The goal is to maximise **Function 6**.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as s
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from itertools import combinations
from mpl_toolkits.mplot3d import Axes3D
from sklearn.exceptions import ConvergenceWarning

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel
warnings.filterwarnings("ignore", category=ConvergenceWarning)

<h1 style="color:#0000CD;font-size:19px;"">Week 1</h1>

**1.1 - Extraction of Initial Data**

In [2]:
inputs = np.load('Initial Data/function_6/initial_inputs.npy')
outputs = np.load('Initial Data/function_6/initial_outputs.npy')
print(inputs.shape, outputs.shape)

(20, 5) (20,)


In [3]:
data = pd.DataFrame(inputs, columns=['x1','x2','x3','x4','x5'])
data['y'] = outputs
display(data)

,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [4]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.7142649478202404 1.8569046837878829


**1.2 - Optimisation (GP + UCB)**

With only 20 initial observations, the global structure and modality of the objective remain unknown. A Gaussian process with an RBF-based covariance
kernel is therefore used as a probabilistic surrogate. Upper Confidence Bound (UCB) is used to balance predicted objective value against posterior
uncertainty during the initial exploration phase.

In [5]:
# Full 5D Cartesian candidate grid
x1 = np.linspace(0,1,25)
x2 = np.linspace(0,1,25)
x3 = np.linspace(0,1,25)
x4 = np.linspace(0,1,25)
x5 = np.linspace(0,1,25)

xx1, xx2, xx3, xx4, xx5 = np.meshgrid(x1, x2, x3, x4, x5)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel(), xx5.ravel()])
del xx1, xx2, xx3, xx4, xx5

# Predict GP mean and uncertainty
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound
kappa = 1.5
ucb = y_pred + kappa * sigma
index_max = np.argmax(ucb)
next_point = np.array(X_grid[index_max], float)
print(f"Next point (with GP+UCB):{next_point[0]:.6f}-{next_point[1]:.6f}-{next_point[2]:.6f}-{next_point[3]:.6f}-{next_point[4]:.6f}")

Next point (with GP+UCB):0.375000-0.333333-0.500000-0.750000-0.125000


<h1 style="color:#0000CD;font-size:19px;"">Week 2</h1>

**2.1 - Previous Week's Query Result**

In [6]:
# New Query Point
x_new = np.array([[0.375000, 0.333333, 0.500000, 0.750000, 0.125000]])
y_new = -0.2654392514813675

def add_QueriedPoint(data, x_new, y_new):

    inputs = data[['x1', 'x2','x3', 'x4','x5']].to_numpy()
    outputs = data['y'].to_numpy()
    
    # Checks if New Points is already included in data
    exists = False
    for i in range(inputs.shape[0]):
        if np.allclose(inputs[i], x_new) and np.isclose(outputs[i], y_new):
            exists = True
            break

    # Only adds if it doesn't exist already 
    if not exists:
        inputs = np.vstack([inputs, x_new])
        outputs = np.append(outputs, y_new)
        data = pd.DataFrame(inputs, columns=['x1', 'x2','x3', 'x4','x5']).assign(y=outputs)
    
        print("Point added!")
    else:
        print("Point already exists, skipping addition.")
    return data

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [7]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.2654392514813675 2.305730380126756


**2.2 - Next Points Queried**

In [8]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 1.5
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.25       0.45833333 0.20833333 0.79166667 0.        ]


<h1 style="color:#0000CD;font-size:19px;"">Week 3</h1>

**3.1 - Previous Week's Query Result**

In [9]:
# New Query Point
x_new = np.array([[0.25000, 0.458333, 0.208333, 0.791667, 0.000]])
y_new = -0.9014136020791392

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [10]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.2654392514813675 2.305730380126756


**3.2 - Next Points Queried**

In [11]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.29166667 0.04166667 0.66666667 1.         0.20833333]


<h1 style="color:#0000CD;font-size:19px;"">Week 4</h1>

**4.1 - Previous Week's Query Result**

In [12]:
# New Query Point
x_new = np.array([[0.291667, 0.041667 , 0.666667 , 1.0, 0.208333]])
y_new = -0.8013803681113528

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [13]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.2654392514813675 2.305730380126756


**4.2 - Next Point Query Selection: UCB with Cartesian-Grid**

In [14]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
next_point = next_point_ucb
print(f"Next point (UCB with Cartesian Grid): {next_point}")

Next point (UCB with Cartesian Grid): [0.29166667 0.04166667 0.625      0.45833333 0.        ]


**Methodological transition.** The initial UCB search evaluated acquisition values over a $25^5$ Cartesian grid. Although this provides systematic domain coverage, its computational cost scales exponentially with dimension and the candidate resolution remains relatively coarse.

From this round onward, the Cartesian grid is replaced by a scrambled Sobol low-discrepancy sequence. The GP kernel is also changed from RBF to Matérn-5/2, which imposes a less restrictive smoothness assumption on the unknown objective.

**4.3 - Next Point Alternative Query Selection: UCB with Sobol Candidates**

In [15]:
from scipy.stats import qmc

d = inputs.shape[1]
m = 13
sampler = qmc.Sobol(d=d, scramble=True, seed=42)
X_cand = sampler.random_base2(m)

# GP Fit First
kernel = Matern(length_scale=[0.2]*d, nu=2.5) + WhiteKernel(noise_level=1e-5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_cand, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_cand[index_ucb]

# Continue exploration using UCB
next_point = next_point_ucb
print(f"Next point (UCB with Sobol Grid): {next_point}")
print("Selected Next point: Continue UCB exploration with Sobol candidate selection")

Next point (UCB with Sobol Grid): [0.44161868 0.44804557 0.85187701 0.85474784 0.02870226]
Selected Next point: Continue UCB exploration with Sobol candidate selection


<h1 style="color:#0000CD;font-size:19px;"">Week 5</h1>

**5.1 - Previous Week's Query Result**

In [16]:
# New Query Point
x_new = np.array([[0.441619, 0.448046, 0.851877, 0.854748, 0.028702]])
y_new = -0.44142971744878756

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [17]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.2654392514813675 2.305730380126756


**5.2. Next Query: EI/UCB Candidate Comparison**

Local refinement phase: 
- Candidate generation is restricted to a local hyperrectangle centred on the current best-observed point, with radii $(0.12, 0.12, 0.12, 0.12, 0.08)$.
- A scrambled Sobol design of $2^{14}$ candidates is evaluated using a Gaussian process with a Constant × Matérn-5/2 covariance kernel and an estimated white-noise component.
- Expected Improvement and UCB are compared over the same candidate set. The UCB exploration coefficient is gradually reduced as the sequential search progresses.
- Week 5 retains a wider minimum separation from existing observations, while subsequent rounds permit closer candidates to support local refinement.

In [18]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(5)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("Continue UCB Exploration")

Current best point: [0.375    0.333333 0.5      0.75     0.125   ]
Current best value: -0.2654392514813675
Next point (EI): [0.46485709 0.29422748 0.55009703 0.7580781  0.04681311]
Next point (UCB): [0.49033749 0.23668618 0.5449834  0.72308624 0.04609319]
Continue UCB Exploration


<h1 style="color:#0000CD;font-size:19px;"">Week 6</h1>

**6.1 - Previous Week's Query Result**

In [19]:
# New Query Point
x_new = np.array([[0.490337, 0.236686, 0.544983, 0.723086, 0.046093]])
y_new = -0.43776384136705904

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [20]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.2654392514813675 2.305730380126756


**6.2. Next Query: EI/UCB Candidate Comparison**

In [21]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(6)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("Transition with EI refinement")

Current best point: [0.375    0.333333 0.5      0.75     0.125   ]
Current best value: -0.2654392514813675
Next point (EI): [0.44122699 0.36679673 0.6129671  0.84554888 0.19416394]
Next point (UCB): [0.46069366 0.38825407 0.6196879  0.78687687 0.20229432]
Transition with EI refinement


<h1 style="color:#0000CD;font-size:19px;"">Week 7</h1>

**7.1 - Previous Week's Query Result**

In [22]:
# New Query Point
x_new = np.array([[0.441227, 0.366797, 0.612967, 0.845549, 0.194164]])
y_new = -0.25167181450647397

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [23]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

-0.25167181450647397 2.3194978171016496


**7.2. Next Query: EI/UCB Candidate Comparison and Selected Query**

In [24]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(7)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI and UCB produce the same candidate - We continue with EI for consistency")

Current best point: [0.441227 0.366797 0.612967 0.845549 0.194164]
Current best value: -0.25167181450647397
Next point (EI): [0.44650066 0.3956494  0.64107547 0.82361719 0.19742492]
Next point (UCB): [0.44650066 0.3956494  0.64107547 0.82361719 0.19742492]
EI and UCB produce the same candidate - We continue with EI for consistency


<h1 style="color:#0000CD;font-size:19px;"">Week 8</h1>

**8.1 - Previous Week's Query Result**

In [25]:
# New Query Point
x_new = np.array([[0.446501, 0.395649, 0.641075, 0.823617, 0.197425]])
y_new = -0.21961390149243976

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


**8.2. Next Query: EI/UCB Candidate Comparison**

In [26]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(8)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("Continue with EI")

Current best point: [0.446501 0.395649 0.641075 0.823617 0.197425]
Current best value: -0.21961390149243976
Next point (EI): [0.42540929 0.36783741 0.7424368  0.70393183 0.20323619]
Next point (UCB): [0.39708783 0.35222125 0.74469906 0.71073035 0.22018971]
Continue with EI


<h1 style="color:#0000CD;font-size:19px;"">Week 9</h1>

**9.1 - Previous Week's Query Result**

In [27]:
# New Query Point
x_new = np.array([[0.425409, 0.367837, 0.367837, 0.703932, 0.203236]])
y_new = -0.5011443246323593

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


**Historical query-record discrepancy:** The Week 8 acquisition output proposed the EI candidate $(0.425409,\ 0.367837,\ 0.742437,\ 0.703932,\ 0.203236)$, whereas the subsequently evaluated record contains $(0.425409,\ 0.367837,\ 0.367837,\ 0.703932,\ 0.203236)$. The evaluated observation is retained as the source of truth because it represents the realised experiment used in all subsequent model updates. No retroactive substitution has been made.

**9.2. Next Query: EI/UCB Candidate Comparison and Selected Query EI**

In [28]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(9)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI and UCB produce the same candidate - We continue with EI for consistency")

Current best point: [0.446501 0.395649 0.641075 0.823617 0.197425]
Current best value: -0.21961390149243976
Next point (EI): [0.44428248 0.38148753 0.60997651 0.94074435 0.16870382]
Next point (UCB): [0.44428248 0.38148753 0.60997651 0.94074435 0.16870382]
EI and UCB produce the same candidate - We continue with EI for consistency


<h1 style="color:#0000CD;font-size:19px;"">Week 10</h1>

**10.1 - Previous Week's Query Result**

In [29]:
# New Query Point
x_new = np.array([[0.444282, 0.381488, 0.609977, 0.940744, 0.168704]])
y_new = -0.3192006437378646

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


**10.2. Next Query: EI/UCB Candidate Comparison**

In [30]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(10)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI selected for next evaluation")

Current best point: [0.446501 0.395649 0.641075 0.823617 0.197425]
Current best value: -0.21961390149243976
Next point (EI): [0.43381762 0.36268792 0.70655996 0.76914793 0.18466693]
Next point (UCB): [0.4451059  0.36791144 0.67349388 0.78178161 0.17857277]
EI selected for next evaluation


<h1 style="color:#0000CD;font-size:19px;"">Week 11</h1>

**11.1 - Previous Week's Query Result**

In [31]:
# New Query Point
x_new = np.array([[0.433818, 0.362688, 0.706560, 0.769148, 0.184667]])
y_new = -0.2027486368481266

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


**11.2. Next query: EI/UCB Candidate Comparison and Selected Query**

In [32]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(11)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI and UCB coincide: we continue with EI")

Current best point: [0.433818 0.362688 0.70656  0.769148 0.184667]
Current best value: -0.2027486368481266
Next point (EI): [0.35406496 0.40332981 0.65195216 0.72230224 0.11658987]
Next point (UCB): [0.35406496 0.40332981 0.65195216 0.72230224 0.11658987]
EI and UCB coincide: we continue with EI


<h1 style="color:#0000CD;font-size:19px;"">Week 12</h1>

**12.1 - Previous Week's Query Result**

In [33]:
# New Query Point
x_new = np.array([[0.354065, 0.403330, 0.651952, 0.722302, 0.116590]])
y_new = -0.10340616640954807

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


**12.2. Next query: EI/UCB Candidate Comparison and Selected Query**

In [34]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(12)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI Continuation")

Current best point: [0.354065 0.40333  0.651952 0.722302 0.11659 ]
Current best value: -0.10340616640954807
Next point (EI): [0.32252688 0.37739632 0.65972528 0.64783988 0.04025006]
Next point (UCB): [0.36538643 0.39844744 0.65080931 0.6633188  0.08425437]
EI Continuation


<h1 style="color:#0000CD;font-size:19px;"">Week 13</h1>

**13.1 - Previous Week's Query Result**

In [35]:
# New Query Point
x_new = np.array([[0.322527, 0.377396, 0.659725, 0.647840, 0.040250]])
y_new = -0.3902352204867967

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


**13.2. Next query: EI/UCB Candidate Comparison and Selected Query**

In [36]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.08])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.001]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.001
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(13)   # decreasing exploration schedule
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)
print("EI Continuation")

Current best point: [0.354065 0.40333  0.651952 0.722302 0.11659 ]
Current best value: -0.10340616640954807
Next point (EI): [0.24191422 0.41742019 0.67290752 0.76165416 0.14512852]
Next point (UCB): [0.24191422 0.41742019 0.67290752 0.76165416 0.14512852]
EI Continuation


<h1 style="color:#0000CD;font-size:19px;"">14. Final Result</h1>

**14.1 -  Previous Week's Query Result**

In [37]:
# Queried Point result
x_new = np.array([[0.241914, 0.417420, 0.672908, 0.761654, 0.145129]])
y_new = -0.3027449686384462

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2', 'x3', 'x4', 'x5']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


Across 13 sequential evaluations, the best observed objective improved from
-0.714265 in the initial design to -0.103406. This represents an improvement of
0.610859 objective units.

The best observed input was:

$
x^\star =
(0.354065,\ 0.403330,\ 0.651952,\ 0.722302,\ 0.116590).
$

The final two evaluations did not improve the incumbent. This indicates a
plateau in the observed best value over the final rounds, but it should not be
interpreted as proof of convergence or global optimality.